# Agentic RAG- RAG That Thinks

**Module 8 · Lesson 8.10**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Query Router - Direct Answer vs Retrieval vs Web
- Self-Correcting Retrieval - Grade, Then Retry
- RAG as a Tool - The Agent Chooses When to Use It
- Multi-Source Router - Pick the Right Knowledge Base
- Self-RAG, CRAG, Adaptive RAG - The Research Patterns
- LangGraph Agentic RAG - The CRAG Loop as a State Machine

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy langchain langgraph llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Question That Didn’t Need a Search

> 💡 **Info**
>
> A user asks your support bot “What’s 2+2?” Standard RAG does what it always does: it embeds the query, searches the vector store, pulls the three “most similar” chunks (a refund clause, a pricing row, a schedule), stuffs them into the prompt, and asks the model to answer “from context.” The model says “4” - but you just paid for an embedding call, a vector search, and a bloated prompt full of irrelevant policy text that could have *misled* the answer. The pipeline has no OFF switch. **Agentic RAG adds one: the LLM decides, per query, whether to retrieve at all, which source to hit, and whether the results are actually good enough - retrying with a better query when they aren’t.** This lesson builds that judgment from scratch, then tours the 2026 patterns (CRAG, LangGraph, multi-agent) that make it production-grade.

### 🎯 What you will be able to do after this lesson

- **Build** agentic RAG by hand - a query router (direct/docs/web), self-correcting retrieval (retrieve→grade→retry), RAG exposed as a tool the agent chooses, and a multi-source router.

- **Compare** the research patterns (Self-RAG, CRAG, Adaptive-RAG, FLARE, IRCoT) and know which to reach for, and in what order.

- **Operate** the 2026 stack: LangGraph StateGraph CRAG loops, LlamaIndex RouterQueryEngine and agents, and multi-agent architectures.

- **Evaluate** production agentic RAG: cost controls (model tiering, caching, iteration limits), circuit breakers, RAGAS agent metrics, and India regulatory routing.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1 (embeddings + retrieval) and 8.8 (conversational RAG), recalled below, a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai numpy` for the by-hand steps (plus `langgraph langchain-google-genai` and `llama-index` for the stack steps). Every block uses the current **google-genai** SDK: pre-2025 tutorials import the retired `google.generativeai` and call dead models and a dead function-calling API, and error on today’s stack.

## 60-Second Warm-Up: What You Already Know

Three recalls, all load-bearing today. Click a card to check yourself.

> 🤖 **Analogy**
>
> **Standard RAG is a librarian who walks to the shelves even when you ask what time it is.** Agentic RAG is a smart librarian who THINKS first: “Do I even need to look this up? Which section? Is what I found actually relevant, or should I search again with different words?” The agent decides to skip retrieval, route to the right source, and self-correct bad results - instead of blindly fetching on every question.
> **Where the analogy breaks down:** a human librarian’s judgment is nearly free and always terminates; an LLM’s “thinking” is an extra call that costs money and latency, and a librarian who keeps second-guessing can loop forever. So agentic RAG is not “always think harder” - it is *think just enough*: every decision (route, grade, retry) is itself an LLM call you must justify, and every loop needs a hard stop. Judgment that never terminates is worse than no judgment.

> 💡 **Info**
>
> ⚠️ Misconception: “agentic RAG means more loops and more agents is always better”
> Wrong, and it’s the expensive mistake. Every decision the agent makes (route? grade? retry? which agent?) is another LLM call, and a multi-agent system is materially more expensive and slower than a single agent for a small quality gain. Judgment is a cost, not a free upgrade. The production discipline is the opposite of “add more”: start with a single agent plus a few tools, add a router only when you have multiple real sources, add self-correction only where retrieval genuinely fails, and go multi-agent only when context windows or heterogeneous sources force it. And every loop needs a hard cap - without one, an agent that keeps re-retrieving similar documents (retrieval thrash) burns your budget forever while making no progress.

> 💡 **Info**
>
> ⛔ Anti-pattern: a self-correction loop with no stop condition
> The tempting **wrong way** to add self-correction is to let the agent keep grading and re-retrieving “until the results are good.” It **fails because** an LLM will happily rephrase forever, and it **breaks when** the rephrasings are all subtly the same query - *retrieval thrash*, where each loop pulls back near-identical documents and makes no progress while the bill climbs. Don’t do this: bound every loop with a hard retry cap AND a content-aware stop (if a new retrieval adds nothing, quit), and give the agent an honest “I couldn’t find it” exit. A loop that cannot terminate is worse than no loop at all.

## Build One: Agentic RAG by Hand

Steps 1–4 build agentic RAG from scratch with nothing but google-genai: a router that decides whether to retrieve, self-correction that grades and retries, RAG exposed as one tool the agent chooses, and a multi-source router. Building it once is what makes every later framework (CRAG, LangGraph, LlamaIndex agents) legible - they are industrial versions of these four moves.

> 📦 **Info**
>
> Standard RAG vs Agentic RAG
> - **Standard:** always retrieves → always augments → always generates. No judgment. Wastes tokens on trivial questions and can inject noise.
> - **Agentic:** the LLM DECIDES whether to retrieve, which source to query, and whether results are sufficient. It can retry, reroute, or answer directly from knowledge.

---

## 🎯 Concept 1: Query Router - Direct Answer vs Retrieval vs Web

### Query Router - Direct Answer vs Retrieval vs Web

The agent classifies each query: answer from knowledge, search docs, or search the web.

**Why this is step 1:** the single cheapest agentic upgrade is an OFF switch for retrieval. Most production traffic is a mix of trivial (“hi”, “2+2”), internal (“refund policy?”), and external (“latest Python version?”) queries. One classification call routes each to the right path - and skipping retrieval on the trivial ones saves real tokens and avoids injecting noise into simple answers.

> 🚃 **Analogy**
>
> **A query router is a receptionist directing visitors.** Not everyone who walks in needs the archive in the basement; the receptionist asks “what do you need?” and points you to the right floor - or answers a quick question at the desk. The router reads the query and sends it to DIRECT (answer now), DOCS (internal store), or WEB (the internet).
> **Where the analogy breaks down:** a receptionist rarely sends you to the wrong floor; an LLM classifier misroutes, and a misroute is costly - route a policy question to DIRECT and the model confidently invents a policy. So production routers add a safety default (when unsure, retrieve rather than guess) and log misroutes, because the failure mode of “confidently answered without looking” is worse than an unnecessary search.

A user types “What’s 2+2?” Standard RAG retrieves three policy chunks and answers from them. What’s the risk?

- **route** asks the LLM to classify the query into DIRECT / DOCS / WEB and returns just the label.

- An unknown label defaults to DOCS - the safe fallback is to retrieve, not to guess.

- Downstream, DIRECT skips retrieval entirely; DOCS hits the vector store; WEB calls a search API.

**📝 `01_query_router.py`** - *google-genai*

In [ ]:
# QUERY ROUTER - the agent decides: answer directly, search docs, or search the web
# pip install google-genai
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

class QueryRouter:
    def route(self, query):
        prompt = (
            "Classify the query into ONE category. Return ONLY the category name.\n"
            "- DIRECT: general knowledge, math, greetings (no retrieval needed)\n"
            "- DOCS: company-specific info, policies, pricing (search internal docs)\n"
            "- WEB: current events, real-time data (search the web)\n\n"
            f"Query: {query}\nCategory:")
        label = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt).text.strip().upper().split()[0]
        return label if label in ("DIRECT", "DOCS", "WEB") else "DOCS"   # default: retrieve, don't guess

router = QueryRouter()
tests = [("What is 2+2?", "DIRECT"), ("What is the refund policy?", "DOCS"),
         ("Who won the IPL match yesterday?", "WEB"), ("Hello!", "DIRECT"),
         ("How much is the GenAI course?", "DOCS")]
correct = 0
for q, expected in tests:
    r = router.route(q); correct += (r == expected)
    print(f"  {'ok' if r==expected else 'XX'}  {q[:34]:34} -> {r}")
print(f"\n  routed {correct}/{len(tests)}; DIRECT skips retrieval (saves tokens + avoids noise)")

# Output:
#   ok  What is 2+2?                       -> DIRECT
#   ok  What is the refund policy?         -> DOCS
#   ok  Who won the IPL match yesterday?   -> WEB
#   ok  Hello!                             -> DIRECT
#   ok  How much is the GenAI course?      -> DOCS
#   routed 5/5; DIRECT skips retrieval (saves tokens + avoids noise)

#### 💡 What just happened

⚡ What just happened? One classification call routed each query: “2+2” and “Hello!” to DIRECT (no retrieval), “refund policy” to DOCS, “IPL match” to WEB. Skipping retrieval on trivial queries is the cheapest agentic win. The tradeoff: the router itself is an LLM call, so it only pays off when a meaningful share of your traffic doesn’t need retrieval - and a misroute (policy question sent to DIRECT) is dangerous, which is why the default is to retrieve when unsure.

- Pick a query and watch the agent classify it to DIRECT / DOCS / WEB, then take the matching path.
- Slide the routing confidence threshold and see borderline queries fall back to retrieval.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Self-Correcting Retrieval - Grade, Then Retry

### Self-Correcting Retrieval - Grade, Then Retry

The agent checks retrieval quality. If results are irrelevant, it rephrases and tries again.

**Why this is step 2:** “most similar” is not “actually relevant.” A query like “can I pay in installments?” may not surface the “EMI via Razorpay” chunk on the first try. Self-correction adds a quality gate: retrieve, have the LLM grade whether the docs answer the question, and if not, rephrase and retry. This is CRAG in miniature - the single most impactful reliability upgrade.

> 🔎 **Analogy**
>
> **Self-correction is a researcher who checks whether the book actually answers the question before quoting it.** A careless researcher grabs the first book off the shelf and cites it; a good one skims it, says “this isn’t quite it,” and searches again with better terms. The grade step is that skim; the rephrase-and-retry is the second search.
> **Where the analogy breaks down:** a human researcher eventually gives up or finds the answer; an LLM can rephrase forever, and worse, it might rephrase into queries that are all subtly the same (retrieval thrash). So self-correction needs a hard retry cap and an honest “I couldn’t find it” exit - a grader that never accepts and never quits is just an expensive infinite loop.

The agent grades retrieval as irrelevant and rephrases the query. How many times should it be allowed to retry?

- **_retrieve** ranks chunks by cosine similarity (Lesson 8.1).

- **_is_relevant** is the LLM grader: does this context actually answer the question? YES/NO.

- **query** loops up to `max_retries`: retrieve → grade → if bad, rephrase and try again; else generate. It always terminates with a hard cap.

**📝 `02_self_correcting.py`** - *google-genai*

In [ ]:
# SELF-CORRECTING RETRIEVAL - grade the results, rephrase and retry if bad (CRAG in miniature)
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class SelfCorrectingRAG:
    def __init__(self, docs, max_retries=2):
        self.docs, self.max_retries = docs, max_retries
        self.embs = [embed(d) for d in docs]

    def _retrieve(self, query, k=2):
        qe = embed(query)
        order = sorted(range(len(self.docs)), key=lambda i: -float(np.dot(qe, self.embs[i])))
        return [self.docs[i] for i in order[:k]]

    def _is_relevant(self, query, docs):
        verdict = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Do these docs answer the query? YES or NO only.\nQuery: {query}\nDocs: {docs}").text.upper()
        return "YES" in verdict

    def _rephrase(self, query):
        return client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Rephrase this query differently for better search. Return ONLY the query.\n{query}").text.strip()

    def query(self, question):
        q = question
        for attempt in range(self.max_retries + 1):        # hard cap - always terminates
            docs = self._retrieve(q)
            if self._is_relevant(question, docs):
                ans = client.models.generate_content(model="gemini-2.5-flash",
                    contents=f"Answer from context only.\n{docs}\nQ: {question}").text.strip()
                return {"answer": ans, "attempts": attempt + 1, "status": "found"}
            if attempt < self.max_retries:
                q = self._rephrase(question)                # self-correct and retry
        return {"answer": "I could not find relevant information.", "status": "exhausted"}

rag = SelfCorrectingRAG([
    "Refund: full within 7 days, 50% for 8-30 days, none after 30.",
    "GenAI course: 14999 rupees. EMI available via Razorpay.",
    "Prerequisites: basic Python, high-school math."])
for q in ["Can I pay in installments?", "Do you teach blockchain?"]:
    r = rag.query(q)
    print(f"Q: {q}\n  [{r['status']}] attempts={r.get('attempts','-')} -> {r['answer'][:60]}\n")

# Output:
# Q: Can I pay in installments?
#   [found] attempts=2 -> Yes - EMI is available via Razorpay ...
# Q: Do you teach blockchain?
#   [exhausted] attempts=- -> I could not find relevant information.

#### 💡 What just happened

⚡ What just happened?“Can I pay in installments?” didn’t match “EMI via Razorpay” at first, so the grader said NO, the agent rephrased, retried, and found it. “Do you teach blockchain?” had no matching doc, so after the retry cap the agent honestly said it couldn’t find it - instead of hallucinating a blockchain course. The tradeoff: each grade and rephrase is an extra LLM call, so self-correction trades cost for reliability, and the hard cap is what keeps a bad query from looping forever.

- Step through retrieve → grade → retry and watch relevance climb (or the retry cap trip).
- Raise the relevance threshold and see more queries need a second attempt.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: RAG as a Tool - The Agent Chooses When to Use It

### RAG as a Tool - The Agent Chooses When to Use It

RAG becomes one of several tools the agent can call, alongside a calculator, a clock, and more.

**Why this is step 3:** the router in step 1 was a hand-written classifier; the modern way is to give the LLM the tools and let it decide. You register Python functions (search docs, calculate, get the time) and the model calls the right one - or none - per query. RAG stops being THE pipeline and becomes ONE tool. This is the architecture behind every production assistant.

> 🧰 **Analogy**
>
> **Tools are a Swiss Army knife the agent reaches into.** You don’t use the scissors for every task - you pick the blade that fits. The agent reads the query, looks at its tools (each with a name and a docstring), and picks the one that fits: search_docs for policy, calculate for GST, get_time for the clock, nothing for “hi.”
> **Where the analogy breaks down:** a Swiss Army knife’s tools are obvious by shape; an agent picks tools by their NAME and DESCRIPTION, so a vague docstring (“does stuff”) makes it grab the wrong blade. Tool-selection accuracy is mostly a function of how well you write the tool descriptions - the agent is only as good as the labels on the knife.

You give the agent three tools. What most determines whether it picks the RIGHT one?

- Plain Python functions with type hints + docstrings ARE the tools - the docstring is what the agent reads to choose.

- **client.chats.create(config=GenerateContentConfig(tools=[...]))** registers them; automatic function calling runs the chosen function for you.

- The model decides per message: call a tool (and the SDK executes it), or answer directly.

**📝 `03_rag_as_tool.py google-genai +`** - *AFC*

In [ ]:
# RAG AS A TOOL - the agent chooses when to search, compute, or answer directly
# pip install google-genai numpy
import numpy as np
from google import genai
from google.genai import types

client = genai.Client()
docs = ["GenAI course: 14999 rupees, 146 hours, 14 modules.",
        "Refund: full within 7 days, 50% for 8-30 days, none after 30.",
        "Live classes 7 PM IST daily. EMI via Razorpay."]
embs = [np.array(client.models.embed_content(model="gemini-embedding-001", contents=d).embeddings[0].values) for d in docs]

def search_docs(query: str) -> str:
    """Search the Netsetos knowledge base for company policies, pricing, and schedules."""
    qe = np.array(client.models.embed_content(model="gemini-embedding-001", contents=query).embeddings[0].values)
    order = sorted(range(len(docs)), key=lambda i: -float(np.dot(qe, embs[i])))
    return " | ".join(docs[i] for i in order[:2])

def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. '14999 * 1.18'."""
    try: return str(eval(expression, {"__builtins__": {}}))   # demo only; use a safe parser in prod
    except Exception: return "Error"

def get_current_time() -> str:
    """Return the current date and time in IST."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M IST")

# register the Python functions as tools; automatic function calling runs them for you
# (minimal example - add request timeouts, retries, and locked-down tools in production)
chat = client.chats.create(model="gemini-2.5-flash", config=types.GenerateContentConfig(
    tools=[search_docs, calculate, get_current_time],
    system_instruction="You are a Netsetos assistant. Use a tool when it helps. Answer concisely."))

for q in ["What is the refund policy?", "How much is 14999 with 18% GST?", "What time is it?", "Hi there!"]:
    print(f"You: {q}\nBot: {chat.send_message(q).text.strip()[:90]}\n")

# Output:
# You: What is the refund policy?           <- agent calls search_docs
# Bot: Full refund within 7 days, 50% for 8-30 days, none after 30.
# You: How much is 14999 with 18% GST?      <- agent calls calculate
# Bot: 14999 with 18% GST is 17,698.82.
# You: What time is it?                    <- agent calls get_current_time
# Bot: It is 2026-07-04 15:12 IST.
# You: Hi there!                            <- no tool, direct reply
# Bot: Hi! How can I help you today?

#### 💡 What just happened

⚡ What just happened? You never wrote routing logic - you registered three Python functions and the agent chose. Refund → `search_docs`, GST math → `calculate`, clock → `get_current_time`, greeting → no tool. RAG is now just one tool among several, and the LLM plans the action. The tradeoff: tool selection is only as good as your docstrings, and letting the model run code (`calculate` uses `eval`) is a real risk - the demo sandboxes builtins, but production needs a safe math parser and locked-down tools.

- Fire a query and watch the agent choose search_docs / calculate / get_time / none.
- Toggle a tool off and see the agent re-plan around what’s available.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Multi-Source Router - Pick the Right Knowledge Base

### Multi-Source Router - Pick the Right Knowledge Base

Route to different vector stores by topic: FAQ, billing, technical.

**Why this is step 4:** one giant index mixes billing, technical, and FAQ content, so a billing query retrieves noisy technical chunks. Splitting into topic-specific stores and routing each query to the right one raises precision - the same routing idea as step 1, now choosing among SOURCES rather than DIRECT/DOCS/WEB.

> 📚 **Analogy**
>
> **Multi-source routing is a switchboard operator connecting you to the right department.** “Billing? One moment.” You don’t want the billing question answered by the engineering desk. The router reads the query and connects it to FAQ, billing, or technical - each a focused index with less noise than one giant directory.
> **Where the analogy breaks down:** a switchboard has a fixed, small set of departments; add too many sources and the router’s job gets as hard as retrieval itself, and cross-department questions (“what’s the price AND the prerequisites?”) route to only one. Beyond a handful of sources you graduate to query decomposition (step 7’s SubQuestionQueryEngine), which fans one question out to several sources and merges the answers.

Why route to three topic-specific stores instead of searching one big combined index?

- **add_source** registers a named store with a description the router reads.

- **_route** asks the LLM which source best fits the query.

- **query** retrieves only within the chosen source - cleaner context, higher precision.

**📝 `04_multi_source.py`** - *google-genai*

In [ ]:
# MULTI-SOURCE RAG - route each query to the best knowledge base
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class MultiSourceRAG:
    def __init__(self): self.sources = {}
    def add_source(self, name, description, docs):
        self.sources[name] = {"desc": description, "docs": docs, "embs": [embed(d) for d in docs]}

    def _route(self, query):
        listing = "\n".join(f"- {n}: {s['desc']}" for n, s in self.sources.items())
        pick = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Which source best answers the query? Return ONLY its name.\n{listing}\nQuery: {query}").text.strip().lower()
        return next((n for n in self.sources if n.lower() in pick), list(self.sources)[0])

    def query(self, question):
        name = self._route(question); s = self.sources[name]
        qe = embed(question)
        order = sorted(range(len(s["docs"])), key=lambda i: -float(np.dot(qe, s["embs"][i])))
        ctx = "\n".join(s["docs"][i] for i in order[:2])
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Answer from context.\n{ctx}\nQ: {question}").text.strip()
        return {"source": name, "answer": ans}

rag = MultiSourceRAG()
rag.add_source("faq", "General FAQs, schedules, onboarding",
    ["Live classes 7 PM IST daily; recordings post within 2 hours.", "Onboarding: activate your account from the welcome email."])
rag.add_source("billing", "Pricing, payments, refunds, EMI",
    ["GenAI: 14999 rupees. EMI via Razorpay.", "Refund: full 7 days, 50% 8-30 days, none after 30."])
rag.add_source("technical", "Prerequisites, curriculum, tools",
    ["Prerequisites: Python basics, high-school math.", "Tools: Colab, Vertex AI, ChromaDB."])

for q in ["Can I get a refund?", "What tools will I learn?", "When are the live classes?"]:
    r = rag.query(q)
    print(f"Q: {q}\n  [{r['source']}] -> {r['answer'][:60]}\n")

# Output:
# Q: Can I get a refund?
#   [billing] -> Full refund within 7 days, 50% for 8-30 days ...
# Q: What tools will I learn?
#   [technical] -> Google Colab, Vertex AI, and ChromaDB.
# Q: When are the live classes?
#   [faq] -> Live classes run at 7 PM IST daily; recordings post within 2 hours.

#### 💡 What just happened

⚡ What just happened?“Refund?” routed to `billing`, “tools?” routed to `technical`, and each retrieved only within its focused store - cleaner context than one mixed index. The tradeoff: routing adds a classification call, and it only pays off past a couple of genuinely distinct sources; for cross-source questions you need query decomposition, which step 7 covers. Steps 1–4 are the whole agentic toolkit by hand; the rest of the lesson swaps each hand-written part for a battle-tested one.

## The 2026 Agentic-RAG Stack

You now own the moves. Steps 5–10 map them onto the tools you ship with: the CRAG/Self-RAG research patterns, LangGraph’s state-machine loops, LlamaIndex’s router and agents, multi-agent architectures, production cost control, and India-specific regulatory routing. Same ideas - industrial parts.

---

## 🎯 Concept 5: Self-RAG, CRAG, Adaptive RAG - The Research Patterns

### Self-RAG, CRAG, Adaptive RAG - The Research Patterns

Generate-critique-revise. Corrective retrieval with web fallback. Complexity-based routing.

**Why this is step 5:** your step-2 self-correction is a hand-rolled version of named research patterns, and knowing them tells you what to build next. CRAG (a grader that routes to correct/incorrect/ambiguous with a web fallback) is the plug-and-play upgrade; Self-RAG (inline reflection tokens) is the lowest-hallucination but needs fine-tuning; Adaptive-RAG routes by query complexity. This step makes step 2 runnable as real CRAG.

> 🎓 **Analogy**
>
> **The patterns are a skill ladder for the same job: retrieve well.** CRAG is the apprentice who double-checks the book and asks a colleague (the web) when unsure - anyone can hire it tomorrow. Self-RAG is the expert who has internalized the checking so deeply it happens mid-sentence - but you had to send it to school (fine-tuning) first. Adaptive-RAG is the manager who sizes the effort to the task.
> **Where the analogy breaks down:** a skill ladder implies you always climb it; here you often stop at the bottom rung on purpose. CRAG solves most teams’ problem at plug-and-play cost, and Self-RAG’s fine-tuning is only worth it at scale or for the strictest accuracy bars. FLARE (retrieve when the model’s token confidence dips) needs access to token logits, which many hosted APIs don’t expose - so the “best” pattern is often the one your stack can actually run.

| Pattern | Mechanism | Plug-and-play? | Reach for it when |
|---|---|---|---|
| CRAG | Evaluator → correct / incorrect / ambiguous → web fallback | Yes | First agentic upgrade over any RAG |
| Self-RAG | Inline reflection tokens [Retrieve][IsRel][IsSup][IsUse] | No (fine-tune) | Strictest accuracy, at scale |
| Adaptive-RAG | Complexity classifier → no / single / iterative retrieval | Mostly | Cost control across mixed traffic |
| FLARE | Retrieve when token confidence drops (needs logits) | Partial | Long-form generation |
| IRCoT | Interleave chain-of-thought reasoning + retrieval | Yes | Multi-hop reasoning |

Which of these patterns can you drop onto an EXISTING RAG with no model fine-tuning?

- **_grade** is the CRAG evaluator: one call labels the retrieval CORRECT / INCORRECT / AMBIGUOUS.

- The branch: CORRECT uses the docs; INCORRECT falls back to web; AMBIGUOUS uses both.

- No fine-tuning - it wraps any retriever. That is why CRAG is the plug-and-play first upgrade.

**📝 `05_crag.py`** - *google-genai*

In [ ]:
# CRAG - Corrective RAG: grade the retrieval, then refine / web-fallback / both
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class CRAG:
    def __init__(self, docs): self.docs, self.embs = docs, [embed(d) for d in docs]
    def _retrieve(self, q, k=2):
        qe = embed(q); order = sorted(range(len(self.docs)), key=lambda i: -float(np.dot(qe, self.embs[i])))
        return [self.docs[i] for i in order[:k]]
    def _grade(self, q, docs):                          # the CRAG evaluator - one call
        r = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Do these docs answer the query? Reply ONE word: CORRECT, INCORRECT, or AMBIGUOUS.\nQ: {q}\nDocs: {docs}").text.strip().upper()
        return r.split()[0] if r.split() else "AMBIGUOUS"
    def _web(self, q): return f"[web result for '{q}']"   # plug in Tavily/Serper in production
    def answer(self, q):
        docs = self._retrieve(q); grade = self._grade(q, docs)
        if grade == "INCORRECT": context = self._web(q)                       # docs bad -> web
        elif grade == "AMBIGUOUS": context = "\n".join(docs) + "\n" + self._web(q)  # use both
        else: context = "\n".join(docs)                                       # docs good
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Answer from context.\n{context}\nQ: {q}").text.strip()
        return {"grade": grade, "answer": ans}

crag = CRAG(["Refund: full within 7 days, 50% for 8-30 days, none after 30.",
             "GenAI course: 14999 rupees, EMI via Razorpay."])
for q in ["What is the refund window?", "Who won the cricket match last night?"]:
    r = crag.answer(q)
    print(f"Q: {q}\n  grade={r['grade']} -> {r['answer'][:60]}\n")

# Output:
# Q: What is the refund window?
#   grade=CORRECT -> Full refund within 7 days, 50% for 8-30 days ...
# Q: Who won the cricket match last night?
#   grade=INCORRECT -> That is not in the local docs; the web-search fallback supplies it.

#### 💡 What just happened

⚡ What just happened? CRAG graded each retrieval and branched: the refund question’s docs were CORRECT (used as-is); the cricket question’s docs were INCORRECT (fell back to web). This is your step-2 self-correction, formalized into the named pattern with a three-way verdict and a web escape hatch - and it needs no fine-tuning, which is exactly why CRAG is the first agentic upgrade most teams should ship. Self-RAG pushes hallucination lower still, but only after you invest in fine-tuning to teach the model its reflection tokens.

- Slide the relevance score and watch the verdict cross from INCORRECT (web) to AMBIGUOUS (both) to CORRECT (docs).
- See which context each branch sends to the generator.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: LangGraph Agentic RAG - The CRAG Loop as a State Machine

### LangGraph Agentic RAG - The CRAG Loop as a State Machine

StateGraph. Nodes for retrieve/grade/rewrite/generate. Conditional edges. Checkpointing.

**Why this is step 6:** your hand-written retry loop works, but production agentic RAG needs durable state, checkpointing, human-in-the-loop, and a hard recursion cap. LangGraph models the self-correcting loop as an explicit state machine - nodes for retrieve/grade/rewrite/generate, a conditional edge that loops back on bad grades - which is why it became the dominant agentic-RAG framework. The official agentic-RAG build guide lives with **LangGraph** at [github.com/langchain-ai/langgraph](https://github.com/langchain-ai/langgraph).

> 🚄 **Analogy**
>
> **LangGraph is a railway with signals, not a car on an open road.** Your hand-rolled loop is a car - it works, but you’re steering every turn and nothing stops you driving off a cliff. LangGraph lays explicit tracks (nodes) and signals (conditional edges): the train can only go where a track leads, the grade signal routes it to “generate” or back to “rewrite,” and a recursion limit is the buffer at the end of the line.
> **Where the analogy breaks down:** real railways can’t loop a train forever, but a graph edge can - a rewrite→retrieve→grade cycle with no counter still runs until the recursion limit trips. So the state itself carries a `retries` counter and the conditional edge checks it; the graph gives you structure, but YOU still have to write the stop condition. Structure prevents wrong turns, not infinite ones.

In a LangGraph CRAG graph, what actually implements the “retry with a better query” loop?

- **State** is a TypedDict carrying the question, documents, generation, and a `retries` counter.

- Each **node** (retrieve/grade/rewrite/generate) is a function that updates state.

- **add_conditional_edges** after grade routes to generate (good docs) or rewrite (bad, under the cap) - that edge is the CRAG loop.

**📝 `06_langgraph_crag.py`** - *LangGraph*

In [ ]:
# LANGGRAPH AGENTIC RAG - the CRAG self-correcting loop as a StateGraph
# pip install langgraph langchain-google-genai
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash")

class State(TypedDict):
    question: str
    original_question: str
    documents: List[str]
    generation: str
    retries: int

def retrieve(state):
    return {"documents": retriever.invoke(state["question"])}       # your vector store (Lesson 8.1)

def grade(state):                                              # keep docs only if relevant
    verdict = llm.invoke(f"Relevant to '{state['question']}'? YES/NO.\n{state['documents']}").content.upper()
    return {"documents": state["documents"] if "YES" in verdict else []}

def rewrite(state):
    better = llm.invoke(f"Rewrite for better retrieval: {state['question']}").content
    return {"question": better, "retries": state.get("retries", 0) + 1}

def generate(state):
    q = state.get("original_question", state["question"])   # answer the ORIGINAL question, not the rewrite
    if not state["documents"]:                          # exhausted with no relevant docs
        return {"generation": "I could not find relevant information."}
    return {"generation": llm.invoke(f"Answer from context.\n{state['documents']}\nQ: {q}").content}

def decide(state):                                             # the conditional edge = the loop
    if state["documents"]: return "generate"
    return "rewrite" if state.get("retries", 0) < 2 else "generate"   # hard cap

g = StateGraph(State)
for name, fn in [("retrieve", retrieve), ("grade", grade), ("rewrite", rewrite), ("generate", generate)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve"); g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", decide, {"generate": "generate", "rewrite": "rewrite"})
g.add_edge("rewrite", "retrieve"); g.add_edge("generate", END)
app = g.compile()   # add checkpointer=PostgresSaver(...) for durable state + HITL

result = app.invoke({"question": "What is the refund policy?",
                     "original_question": "What is the refund policy?", "retries": 0})
print(result["generation"])

# Output:
# Full refund within 7 days, 50% for 8-30 days, none after 30.

#### 💡 What just happened

⚡ What just happened? The same retrieve→grade→retry loop from step 2, now an explicit state machine: nodes update state, and one conditional edge after `grade` either generates (good docs) or rewrites and re-retrieves (bad docs) - capped by the `retries` counter. That structure is why LangGraph dominates agentic RAG: it makes the loop legible, checkpointable (`PostgresSaver` for durable state and human-in-the-loop), and bounded. The tradeoff vs your hand-rolled loop: more ceremony up front, in exchange for production-grade state, observability, and recovery.

- Step through the graph and watch the token move retrieve → grade → (rewrite → retrieve) → generate.
- See the conditional edge branch on the grade, and the retry counter cap the loop.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: LlamaIndex Agentic RAG - RouterQueryEngine and Agents

### LlamaIndex Agentic RAG - RouterQueryEngine and Agents

Route to the right index in a few lines; decompose cross-document questions.

**Why this is step 7:** LlamaIndex collapses your step-4 multi-source router into a few lines - `RouterQueryEngine` selects among indices for you - and adds `SubQuestionQueryEngine` for the cross-document questions a single router can’t handle. Where LangGraph gives you a general state machine, LlamaIndex gives you RAG-native building blocks.

> 🏪 **Analogy**
>
> **RouterQueryEngine is a pre-built switchboard you rent instead of wiring yourself.** Your step-4 router was hand-soldered; LlamaIndex ships the switchboard - you plug in each index with a description and it routes. And `SubQuestionQueryEngine` is the operator who splits a two-part request (“compare Lyft vs Uber revenue”) into two calls to two departments and merges the answers.
> **Where the analogy breaks down:** a rented switchboard hides its wiring, which is convenient until it misroutes and you can’t see why. LlamaIndex’s selectors are LLM calls with the same misroute risk as your hand-rolled router - the abstraction saves code, not judgment. And the newer `FunctionAgent`/`AgentWorkflow` classes replaced the old `Agent` classes, so pre-2025 tutorials import names that no longer exist.

A query asks to “compare the billing and technical prerequisites.” Can a single `RouterQueryEngine` answer it well?

- Wrap each index as a **QueryEngineTool** with a description the selector reads.

- **RouterQueryEngine** + **LLMSingleSelector** picks one tool per query - Adaptive RAG at the query-engine level.

- For “compare X vs Y” questions, swap in **SubQuestionQueryEngine**, which decomposes and merges.

**📝 `07_llamaindex_agentic.py`** - *LlamaIndex*

In [ ]:
# LLAMAINDEX AGENTIC RAG - route each query to the right index in a few lines
# pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

# billing_index / tech_index are VectorStoreIndex objects over your two doc sets
tools = [
    QueryEngineTool.from_defaults(query_engine=billing_index.as_query_engine(),
        description="Pricing, refunds, EMI, and payment questions."),
    QueryEngineTool.from_defaults(query_engine=tech_index.as_query_engine(),
        description="Prerequisites, curriculum, and tools."),
]
router = RouterQueryEngine(selector=LLMSingleSelector.from_defaults(), query_engine_tools=tools)

print(router.query("Can I pay in installments?"))     # -> routed to the billing tool
# For "compare X vs Y" queries, use SubQuestionQueryEngine to decompose + merge.
# For a full tool-using agent, use FunctionAgent / AgentWorkflow (the old Agent classes are retired).

# Output:
# EMI is available via Razorpay; installment plans are supported.

#### 💡 What just happened

⚡ What just happened? Your step-4 multi-source router became three objects: two `QueryEngineTool`s and a `RouterQueryEngine` that selects between them. That is Adaptive RAG at the query-engine level. For cross-document questions (“compare X vs Y”) you reach for `SubQuestionQueryEngine`, which decomposes into sub-questions, routes each, and merges - the pattern a single router can’t do. The currency tradeoff: LlamaIndex moved to `FunctionAgent`/`AgentWorkflow`, so old tutorials’ `Agent` imports fail; use the new classes.

---

## 🎯 Concept 8: Multi-Agent RAG - Supervisor, Planner-Executor, and When Not To

### Multi-Agent RAG - Supervisor, Planner-Executor, and When Not To

Several specialist agents under a coordinator. Powerful, and easy to over-use.

**Why this is step 8:** a single agent with tools handles most needs, but some jobs genuinely want specialists - a billing agent, a technical agent, a web agent, each with its own sources and even its own model. A supervisor coordinates them. The critical lesson is restraint: multi-agent is markedly more expensive and slower, so it must earn its place.

> 🏛️ **Analogy**
>
> **A supervisor with specialist agents is a company org chart.** A manager (supervisor) reads the request and delegates to the right department (specialist agent); each department has its own filing cabinets (sources) and expertise. It scales to problems one generalist can’t hold in their head.
> **Where the analogy breaks down:** real org charts add people because the work genuinely requires them; teams add agents because it feels sophisticated. Every agent is more LLM calls, more coordination, more latency - and coordination overhead grows fast. So the honest rule is the reverse of “more is better”: stay single-agent until context windows, per-agent models, or heterogeneous parallel retrieval force your hand, then add the fewest agents that solve it.

A single agent with a few tools already answers your queries well. Should you upgrade to multi-agent?

| Architecture | Pattern | Best for | Watch out for |
|---|---|---|---|
| Supervisor | Central coordinator delegates to sub-agents | Enterprise RAG, compliance | Coordinator becomes a bottleneck |
| Planner-executor | Plan the steps, then retrieve + synthesize each | Auditable, per-step trails | Planning overhead |
| Peer-to-peer | Agents talk directly to each other | Cross-domain research | Coordination cost grows fast |

- **supervisor** is one LLM call that picks the specialist agent for a query.

- Each named agent (billing/tech/web) is itself a full RAG agent (steps 1–6) over its own sources.

- Use this only when specialists genuinely differ - otherwise a single agent with tools is cheaper.

**📝 `08_multi_agent.py`** - *Supervisor*

In [ ]:
# MULTI-AGENT RAG - a supervisor routes to specialist agents (use sparingly)
# pip install langchain-google-genai
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash")

def supervisor(query):
    # the supervisor picks ONE specialist; each is a full RAG agent over its own sources
    return llm.invoke(
        "Route to ONE agent: billing_agent, tech_agent, or web_agent. Reply the name only.\n"
        f"Query: {query}").content.strip()

for q in ["Refund policy?", "What tools will I learn?", "Latest Python version?"]:
    print(f"  {q:26} -> {supervisor(q)}")

# Reach for multi-agent ONLY when a single agent + tools cannot cope:
# insufficient context window, per-agent models, or heterogeneous parallel retrieval.

# Output:
#   Refund policy?             -> billing_agent
#   What tools will I learn?   -> tech_agent
#   Latest Python version?     -> web_agent

#### 💡 What just happened

⚡ What just happened? A supervisor delegated each query to a specialist agent - billing, technical, or web - each of which is a full RAG agent (steps 1–6) over its own sources. That scales to problems a single agent can’t hold. But every agent multiplies LLM calls and coordination, so multi-agent buys a modest quality gain at a real cost multiple; the discipline is to stay single-agent until a concrete constraint (context, per-agent models, heterogeneous sources) forces the upgrade. Frameworks: CrewAI for role-based teams, LangGraph for graph control, AutoGen for distributed systems.

---

## 🎯 Concept 9: Production - Cost, Caching, and the Circuit Breaker

### Production - Cost, Caching, and the Circuit Breaker

Model tiering, semantic caching, iteration limits, and a DEGRADED state.

**Why this is step 9:** every agentic decision is an LLM call, so a proof-of-concept that’s cheap can become ruinous at scale - and an agent that loops can run up the bill on a single query. This step is the economics: run cheap models for classification and frontier only for generation, cache aggressively, cap the loops, and add a circuit breaker because a hallucinating LLM still returns HTTP 200.

> ⚡ **Analogy**
>
> **The circuit breaker is the fuse box in your house.** When a circuit draws too much, the breaker trips before the wiring melts. Agentic RAG needs the same: when a query loops too many times, hallucinates, or overruns cost, the breaker trips to a DEGRADED state (fall back to simple RAG, or a cached or static answer) instead of burning money.
> **Where the analogy breaks down:** a home fuse trips on a clean physical signal (too many amps); an agent’s failure is semantic and invisible - the LLM returns a confident 200 while hallucinating, so there’s no current spike to detect. That’s why the DEGRADED state exists: it watches soft signals (loop count, grounding checks, cost per query) that ordinary error handling never sees. You have to build the sensor, not just the switch.

An agentic RAG POC costs $50. What’s the single biggest lever to keep it cheap at scale?

- **route_model** sends grade/route tasks to a cheap tier, generation to the frontier tier.

- **SemanticCache** serves repeated queries from cache instead of re-running the agent.

- **CircuitBreaker** trips CLOSED → DEGRADED → OPEN as the loop count climbs - a hard stop against runaway loops.

**📝 `09_production.py Pure`** - *Python*

In [ ]:
# PRODUCTION AGENTIC RAG - model tiering, semantic caching, and a circuit breaker
import hashlib

def route_model(task):
    # cheap tier for the many classification calls; frontier tier only for generation
    return "gemini-2.5-flash-lite" if task in ("grade", "route") else "gemini-2.5-flash"

class SemanticCache:
    def __init__(self): self.store = {}
    def get_or_run(self, query, fn):
        key = hashlib.sha256(query.lower().encode()).hexdigest()   # demo: exact-match hash
        # a real semantic cache embeds the query and returns a cached answer above a cosine-similarity threshold
        if key in self.store: return self.store[key], True       # cache hit - no agent run
        self.store[key] = fn(query); return self.store[key], False

class CircuitBreaker:
    """CLOSED -> DEGRADED -> OPEN as loops climb. DEGRADED is where you ALSO add hallucination/cost checks (stubbed here)."""
    def __init__(self, max_loops=3): self.max_loops, self.loops, self.state = max_loops, 0, "CLOSED"
    def step(self):
        self.loops += 1
        if self.loops >= self.max_loops: self.state = "OPEN"            # stop - fall back
        elif self.loops >= self.max_loops - 1: self.state = "DEGRADED"
        return self.state

print("grade ->", route_model("grade"), "| generate ->", route_model("generate"))
cache = SemanticCache(); hits = 0
for q in ["refund?", "refund?", "price?"]:
    _, hit = cache.get_or_run(q, lambda s: f"answer({s})"); hits += int(hit)
print(f"cache hits: {hits}/3")
cb = CircuitBreaker(max_loops=3)
print("loop states:", [cb.step() for _ in range(3)])

# Output:
# grade -> gemini-2.5-flash-lite | generate -> gemini-2.5-flash
# cache hits: 1/3
# loop states: ['CLOSED', 'DEGRADED', 'OPEN']

#### 💡 What just happened

⚡ What just happened? Three cost controls in one block: tiering sent grading to a cheap model and generation to the frontier; the cache served the repeated “refund?” without re-running the agent; and the breaker walked CLOSED → DEGRADED → OPEN as loops climbed, giving you a hard stop. The reason the DEGRADED state exists at all: an LLM returns a confident HTTP 200 while hallucinating, so ordinary error handling can’t catch it - you have to watch semantic signals (loops, grounding, cost) and fall back gracefully. These controls are what separate a $50 POC from a bill that doesn’t explode at scale.

- Slide the tiering mix, cache hit rate, and loop cap and watch cost per 1,000 queries move.
- Push loops past the cap and see the circuit breaker trip to DEGRADED, then OPEN.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: India Agentic RAG - Regulatory Routing, Indic Models, Cost

### India Agentic RAG - Regulatory Routing, Indic Models, Cost

GST/IT/SEBI routing, temporal awareness, Sarvam models, sovereign cloud.

**Why this is step 10:** Indian regulatory chatbots are a killer app for agentic RAG - they must route across GST / Income Tax / SEBI / RBI sources, know which rule VERSION applies to which year, answer in the user’s language, and control cost. Getting this right combines multi-source routing (steps 1, 4), temporal awareness, Indic-native models, and sovereign hosting.

> 🇮🇳 **Analogy**
>
> **An Indian regulatory agent is a chartered accountant who knows which year’s rulebook to open.** Tax law isn’t static - a good CA doesn’t just find the GST rule, they find the version that applied to YOUR assessment year, and they answer in the language you asked in. The agent routes to the right Act, picks the right temporal version, and generates in Hindi or English as needed.
> **Where the analogy breaks down:** a CA carries decades of judgment; the agent only knows what’s in its retrieved documents, so temporal routing is only as good as your versioned corpus - miss the amendment and it confidently cites a repealed rule. And regulatory answers carry legal weight, so these systems keep a human in the loop and cite sources, rather than letting the agent’s judgment be the final word.

A user asks a GST question about a past assessment year. What must the agent get right beyond routing to the GST corpus?

> 📦 **Info**
>
> 🌐 The India agentic-RAG stack
> - **Regulatory routing:** detect language → classify intent → route to the domain corpus (GST, Income Tax/CBDT, SEBI, RBI) → verify against cross-references → generate with citations.
> - **Temporal awareness:** which rule version applies to which assessment year (the GST Council meets often; the Income-Tax Act was overhauled). Version your corpus so the agent cites the rule that was in force.
> - **Indic models:** Sarvam AI’s 2026 open-weight releases include **Sarvam 30B** and **Sarvam 105B** (Mixture-of-Experts, reported across many Indian languages and strong at tool-calling, superseding the earlier `sarvam-m`) - the Indic model lineup moves fast, so verify the current models and sizes before committing. Vyakyarth embeddings for Hindi retrieval.
> - **Cost tiering:** a small model for routing (near-zero) + a semantic cache + a self-hosted Indic model for simple queries + a frontier model only for hard multi-hop reasoning - the same tiering as step 9, tuned for the Indian market (verify current pricing).
> - **Sovereign cloud:** Krutrim, Yotta (IndiaAI partners), and hyperscaler India regions for data residency - regulatory data often can’t leave the country.

- **detect_lang** flags Devanagari (Hindi) vs Latin script from the query text.

- **route_domain** keyword-routes to GST / income-tax / SEBI (an LLM classifier in production).

- Downstream, each domain retrieves from its own version-aware corpus and generates with citations.

**📝 `10_india.py Pure`** - *Python*

In [ ]:
# INDIA AGENTIC RAG - language + regulatory-domain routing (cost-optimized)
DOMAINS = {
    "gst": ["gst", "input tax credit", "gstr"],
    "income_tax": ["income tax", "tds", "assessment year"],
    "sebi": ["sebi", "mutual fund", "listing"],
}

def detect_lang(text):
    return "hi" if any("ऀ" <= c <= "ॿ" for c in text) else "en"   # Devanagari range

def route_domain(query):
    q = query.lower()
    for domain, kws in DOMAINS.items():
        if any(k in q for k in kws): return domain
    return "general"

for q in ["GST input tax credit rules?", "रिफंड कैसे मिलेगा?", "SEBI mutual fund norms?"]:
    print(f"  lang={detect_lang(q)}  domain={route_domain(q):11}  | {q}")

# Indic stack: Sarvam open-weight models for generation (verify the current lineup),
# Vyakyarth embeddings for Hindi retrieval, sovereign cloud (Krutrim/Yotta) for residency.

# Output:
#   lang=en  domain=gst          | GST input tax credit rules?
#   lang=hi  domain=general      | रिफंड कैसे मिलेगा?
#   lang=en  domain=sebi         | SEBI mutual fund norms?

#### 💡 What just happened

⚡ What just happened? The router detected language (Devanagari → Hindi) and keyword-routed to the GST or SEBI corpus - the same routing idea from steps 1 and 4, applied to regulatory domains, with temporal versioning layered on so the agent cites the rule that was actually in force. The Indian production pattern stacks it all: a near-free router, a semantic cache, a self-hosted Indic model (Sarvam’s open-weight releases) for the bulk, and a frontier model only for hard reasoning - on sovereign cloud so regulatory data stays in-country. And because answers carry legal weight, a human stays in the loop with citations, rather than the agent’s judgment being final.

## Putting It Together

You gave RAG judgment - a router to decide whether and where to retrieve, a grade-and-retry loop to self-correct, tools the agent chooses among - then toured the stack that hardens it: the CRAG/Self-RAG patterns, LangGraph’s state-machine loops, LlamaIndex’s router and agents, multi-agent architectures, production cost control, and India’s regulatory routing. The through-line: *agentic RAG is a series of decisions the LLM makes, and every decision is a cost you must justify and a loop you must bound - the skill is adding judgment where it pays and stopping it where it doesn’t.*

> 📦 **Info**
>
> 🔑 Key takeaways
> - **Retrieval is a decision, not a reflex.** A router that skips retrieval on trivial queries is the cheapest agentic upgrade; when unsure, default to retrieving.
> - **Grade, then correct.** “Most similar” is not “relevant” - have the LLM grade retrieval and rephrase/retry (CRAG); always cap the loop.
> - **RAG is one tool.** Register functions and let the agent choose; tool-selection accuracy lives in the docstrings.
> - **Pick the pattern your stack can run.** CRAG is plug-and-play; Self-RAG is lowest-hallucination but needs fine-tuning; LangGraph makes the loop a bounded, checkpointable state machine.
> - **Restraint beats sophistication.** Multi-agent is expensive; stay single-agent + tools until a real constraint forces more.
> - **Production is cost + safety.** Model tiering, semantic caching, hard iteration caps, and a DEGRADED circuit-breaker state - because a hallucinating LLM still returns HTTP 200.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - In Lesson 8.11 (**RAG Evaluation**) you measure whether the agent actually helps - RAGAS agent metrics (ToolCallAccuracy, AgentGoalAccuracy) become the golden-set + CI gate on every change.
> - In Module 11 (**AI Agents**) the single-purpose RAG agent grows into general planning, memory, and multi-agent orchestration.
> - In Module 14 (**LLMOps**) the cost controls and circuit breakers become monitored production telemetry - per-query iterations, fallback rate, and cost dashboards.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the agentic-RAG stack.

---

## 🎓 Summary

You've completed the practical part of **Agentic RAG- RAG That Thinks**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_10.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.10.html` - regenerate this notebook whenever the source HTML is updated.*
